In [2]:
from optimize_hyper import OptimizeHyper

import numpy as np

import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import balanced_accuracy_score

import matplotlib.pyplot as plt

from CellsData import CellsData
from CustomDataloader import CustomLoader

from training_utils import model_run, set_seed, stratified_cv_split, get_sub_dataset

from models import (
    MIL_model,
    MLP_encoder,
    MaxAggergation,
    AttentionAggregation,
    GatedAttentionAggregation,
)

from bayes_opt import BayesianOptimization

import yaml
import ast
from datetime import datetime
from typing import Tuple, Union
from itertools import chain


In [4]:
attention_hidden_size= 42.987061617991834
encoding_size= 4.665246790868912
hidden_size= 43.162108397553105
log_decay= -3.0
log_learning_rate= -1.0
n_hidden= 1.56327455585723



n_hidden = int(np.floor(n_hidden))
hidden_size = int(np.floor(hidden_size))
encoding_size = int(np.floor(encoding_size))
attention_hidden_size = int(np.floor(attention_hidden_size))

settings = {
    "encoder": MLP_encoder,
    "encoder_settings": {
        "n_hidden": n_hidden,
        "hidden_size": hidden_size,
        "output_size": encoding_size,
        "input_size": 30,
    },
    "aggregator": AttentionAggregation,
    "aggregator_settings": {
        "encoding_size": encoding_size,
        "attention_hidden_size": attention_hidden_size,
    },
    "learning_rate": 10**log_learning_rate,
    "decay": 10**log_decay,
    "sparse": False,
        }

In [5]:
self = OptimizeHyper(sparse=False, n_epochs=60)

In [17]:
cv_split = 5

losses = []
for seed in self.seeds:
    seed_losses = []
    cv_indices = stratified_cv_split(
        dataset=self.train_set, k_cv=cv_split, seed=seed
    )
    for k in range(cv_split):
        validation_set = []
        for idx in cv_indices[k]:
            validation_set.append(self.train_set[idx])

        train_set_k = [i for i in range(cv_split) if i != k]
        train_set_indices = chain(*[cv_indices[i] for i in train_set_k])
        train_set = []
        for idx in train_set_indices:
            train_set.append(self.train_set[idx])

        loss = self.model_test(
            **settings, train_set=train_set, validation_set=validation_set
        )
        seed_losses.append(loss)

    mean_loss = np.mean(seed_losses)
    print(seed)
    print(seed_losses)
    print(np.std(seed_losses))
    losses.append({"loss": mean_loss, "seed": seed})

losses.sort(key=lambda x: x["loss"])

0
[0.21503803843543642, 0.3452002576419285, 0.2629103830882481, 0.23170146942138672, 0.34061065473054586]
0.05434236600424095
37
[0.2699743778932662, 0.3321725442295983, 0.30151375560533433, 0.29338052272796633, 0.3483894247757761]
0.0279695105742913
42
[0.2956236586684272, 0.30371324221293133, 0.32117222320465816, 0.334158730506897, 0.3240230961849815]
0.014047622146826524


In [18]:
losses

[{'loss': 0.27909216066350917, 'seed': 0},
 {'loss': 0.30908612504638827, 'seed': 37},
 {'loss': 0.315738190155579, 'seed': 42}]